# Proposed Method — Text-Only RoBERTa AES (8 Traits)

**Hyperparameters tetap:** `lr=2e-5`, `dropout=0.2`, `hidden_dim=256`, `epochs=15`.

In [ ]:
import os
import re
import random
import gc
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            torch.xpu.manual_seed(seed)
            torch.xpu.manual_seed_all(seed)
    except ImportError:
        print("Warning: intel_extension_for_pytorch (ipex) tidak ditemukan.")
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

def get_device():
    if torch.cuda.is_available(): return "cuda"
    try:
        if torch.xpu.is_available(): return "xpu"
    except: pass
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

DEVICE = get_device()
print(f"Using Device: {DEVICE}")

Using Device: cuda


## Konfigurasi

Hyperparameter dikunci sesuai best params dari grid search RQ sebelumnya.

In [ ]:
CSV_FILE   = "data.csv"
BATCH_SIZE = 8

# Hyperparameters TETAP (no grid search)
LEARNING_RATE = 2e-5
DROPOUT_RATE  = 0.2
HIDDEN_DIM    = 256
EPOCHS        = 15

TEXT_TRAITS = [
    'organizational_structure(ground_truth)', 'coherence(ground_truth)',
    'essay_length(ground_truth)', 'grammatical_accuracy(ground_truth)',
    'grammatical_diversity(ground_truth)', 'lexical_accuracy(ground_truth)',
    'lexical_diversity(ground_truth)', 'punctuation_accuracy(ground_truth)',
]

BEST_MODEL_SAVE_PATH = 'best_roberta_text.pth'
TRAIN_PRED_CSV = 'roberta_train.csv'
TEST_PRED_CSV  = 'roberta_test.csv'

print(f"LR={LEARNING_RATE} | Drop={DROPOUT_RATE} | HiddenDim={HIDDEN_DIM} | Epochs={EPOCHS}")
print(f"Traits ({len(TEXT_TRAITS)}): {[t.replace('(ground_truth)','') for t in TEXT_TRAITS]}")

LR=2e-05 | Drop=0.2 | HiddenDim=256 | Epochs=15
Traits (8): ['organizational_structure', 'coherence', 'essay_length', 'grammatical_accuracy', 'grammatical_diversity', 'lexical_accuracy', 'lexical_diversity', 'punctuation_accuracy']


## Dataset

In [ ]:
class TextOnlyDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        essay_text  = str(row.get('Essay', ""))
        prompt_text = str(row.get('Question', ""))

        enc = self.tokenizer(
            prompt_text, essay_text,
            max_length=512, padding="max_length", truncation=True, return_tensors="pt"
        )
        labels = [row[c] for c in TEXT_TRAITS]

        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels_txt':     torch.tensor(labels, dtype=torch.float32),
        }

## Model

In [ ]:
class RobertaSemanticAES(nn.Module):
    def __init__(self, dropout_rate=DROPOUT_RATE, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.roberta = AutoModel.from_pretrained("roberta-base")

        for p in self.roberta.parameters():
            p.requires_grad = False
        for layer in self.roberta.encoder.layer[-6:]:
            for p in layer.parameters():
                p.requires_grad = True

        self.text_dim = 768
        self.dropout  = nn.Dropout(dropout_rate)

        self.txt_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.text_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim, 1),
            ) for _ in range(len(TEXT_TRAITS))
        ])

    def cls_pooling(self, model_output):
        # CLS = first token embedding
        return model_output.last_hidden_state[:, 0, :]

    def forward(self, input_ids, attention_mask):
        txt_out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        bert_embed = self.cls_pooling(txt_out)
        bert_embed = self.dropout(bert_embed)
        txt_preds  = [h(bert_embed) for h in self.txt_heads]
        return torch.cat(txt_preds, dim=1)

## Helper: Evaluate QWK

In [ ]:
def evaluate_qwk(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for b in dataloader:
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbl  = b['labels_txt'].cpu().numpy()

            preds = model(ids, mask).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(lbl)

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    qwk_scores = []
    for i in range(len(TEXT_TRAITS)):
        pred_classes = np.round(all_preds[:, i]  * 2).astype(int)
        gt_classes   = np.round(all_labels[:, i] * 2).astype(int)
        try:
            qwk = cohen_kappa_score(gt_classes, pred_classes, weights='quadratic')
            if np.isnan(qwk): qwk = 0.0
        except:
            qwk = 0.0
        qwk_scores.append(qwk)
    return qwk_scores

## STAGE 1 — Data Loading

In [ ]:
try:
    df = pd.read_csv(CSV_FILE, encoding='latin1')
    print(f"Loaded {len(df)} rows dari {CSV_FILE}.")
except Exception as e:
    print(f"CSV not found ({e}). Creating dummy data.")
    df = pd.DataFrame({
        'Essay':    ["This is a test essay with good grammar."] * 60,
        'Question': ["Write a test essay."] * 60,
        'graph':    np.random.randint(1, 4, 60),
    })
    for c in TEXT_TRAITS:
        df[c] = np.random.randint(1, 6, size=len(df))

if os.path.exists('train_df.csv') and os.path.exists('test_df.csv'):
    print("Memuat ../../train_df.csv & ../../test_df.csv (test set konsisten)...")
    train_df = pd.read_csv('train_df.csv')
    test_df  = pd.read_csv('test_df.csv')
elif os.path.exists('train_df.csv') and os.path.exists('test_df.csv'):
    print("Memuat train_df.csv & test_df.csv lokal...")
    train_df = pd.read_csv('train_df.csv')
    test_df  = pd.read_csv('test_df.csv')
else:
    print("File split belum ada. Melakukan GroupShuffleSplit baru...")
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss_test.split(df, groups=df['graph']))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df  = df.iloc[test_idx].reset_index(drop=True)
    train_df.to_csv('train_df.csv', index=False)
    test_df.to_csv('test_df.csv',  index=False)

bert_tok    = AutoTokenizer.from_pretrained("roberta-base")
test_ds     = TextOnlyDataset(test_df, bert_tok)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows.")

Loaded 1054 rows dari data.csv.
Memuat ../../train_df.csv & ../../test_df.csv (test set konsisten)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: 867 rows | Test: 187 rows.


## STAGE 2 — Final Training (100% Train Set, Fixed Hyperparameters)

In [ ]:
set_seed(42)
g_final = torch.Generator(); g_final.manual_seed(42)

full_train_loader = DataLoader(
    TextOnlyDataset(train_df, bert_tok),
    batch_size=BATCH_SIZE, shuffle=True, generator=g_final,
)

if 'final_model' in locals():
    del final_model
if 'final_optimizer' in locals():
    del final_optimizer
gc.collect()
if DEVICE == "xpu":  torch.xpu.empty_cache()
elif DEVICE == "cuda": torch.cuda.empty_cache()

final_model     = RobertaSemanticAES(dropout_rate=DROPOUT_RATE, hidden_dim=HIDDEN_DIM).to(DEVICE)
final_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=LEARNING_RATE,
)
criterion_score = nn.MSELoss()

for epoch in range(EPOCHS):
    final_model.train()
    loop = tqdm(full_train_loader, desc=f"Final Ep {epoch+1}/{EPOCHS}")
    for b in loop:
        final_optimizer.zero_grad()
        ids  = b['input_ids'].to(DEVICE)
        mask = b['attention_mask'].to(DEVICE)
        lbl  = b['labels_txt'].to(DEVICE)

        preds = final_model(ids, mask)
        loss  = criterion_score(preds, lbl)
        loss.backward()
        final_optimizer.step()
        loop.set_postfix({'Loss': f"{loss.item():.4f}"})

torch.save(final_model.state_dict(), BEST_MODEL_SAVE_PATH)
print(f"Final Model Saved -> {BEST_MODEL_SAVE_PATH}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Final Ep 1/15:   0%|          | 0/109 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable d

Final Model Saved -> best_roberta_text_rq4.pth


## STAGE 3 — Extract Predictions & Evaluasi Akhir

In [ ]:
final_model.eval()

full_train_loader_noshuffle = DataLoader(
    TextOnlyDataset(train_df, bert_tok), batch_size=BATCH_SIZE, shuffle=False,
)

roberta_train_preds = []
with torch.no_grad():
    for b in tqdm(full_train_loader_noshuffle, desc="Extracting Train Preds"):
        ids, mask = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)
        preds = final_model(ids, mask).cpu().numpy()
        roberta_train_preds.append(preds)
roberta_train_preds = np.vstack(roberta_train_preds)

roberta_test_preds = []
with torch.no_grad():
    for b in tqdm(test_loader, desc="Extracting Test Preds"):
        ids, mask = b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE)
        preds = final_model(ids, mask).cpu().numpy()
        roberta_test_preds.append(preds)
roberta_test_preds = np.vstack(roberta_test_preds)

trait_short = [t.replace('(ground_truth)', '') for t in TEXT_TRAITS]

df_train_preds = pd.DataFrame(roberta_train_preds, columns=trait_short)
df_test_preds  = pd.DataFrame(roberta_test_preds,  columns=trait_short)
df_train_preds.to_csv(TRAIN_PRED_CSV, index=False)
df_test_preds.to_csv(TEST_PRED_CSV,   index=False)
print(f"Train preds -> {TRAIN_PRED_CSV}")
print(f"Test  preds -> {TEST_PRED_CSV}")

qwk_scores = {}
for short, gt_col in zip(trait_short, TEXT_TRAITS):
    pred_vals    = df_test_preds[short].values
    gt_vals      = test_df[gt_col].values
    pred_classes = np.round(pred_vals * 2).astype(int)
    gt_classes   = np.round(gt_vals   * 2).astype(int)
    qwk_scores[gt_col] = cohen_kappa_score(gt_classes, pred_classes, weights='quadratic')

print("\n=== HASIL QWK PER TRAIT (FINAL MODEL vs TEST SET MURNI) ===")
for trait, sc in qwk_scores.items():
    print(f"  {trait.replace('(ground_truth)',''):<25s} : {sc:.4f}")
mean_qwk = float(np.mean(list(qwk_scores.values())))
print("-" * 40)
print(f"  {'RATA-RATA QWK FINAL':<25s} : {mean_qwk:.4f}")

Extracting Test Preds: 100%|██████████| 24/24 [00:06<00:00,  3.88it/s]

Train preds -> roberta_train_rq4.csv
Test  preds -> roberta_test_rq4.csv

=== HASIL QWK PER TRAIT (FINAL MODEL vs TEST SET MURNI) ===
  organizational_structure  : 0.5286
  coherence                 : 0.6356
  essay_length              : 0.5085
  grammatical_accuracy      : 0.6594
  grammatical_diversity     : 0.5641
  lexical_accuracy          : 0.5810
  lexical_diversity         : 0.5996
  punctuation_accuracy      : 0.6248
----------------------------------------
  RATA-RATA QWK FINAL       : 0.5877
